<h1 align="center"> VIRTUAL INTERNSHIP - 1</h1>

<h3 align="center">📊 Data Analysis & Business Insights</h3>

<h3 align="left">📊 Dataset Structure & Data Model</h3>

<p align="center">
  <b>Nova Mart — Campaign Performance Analysis</b>
</p>

<hr>

## 🗂️ 1. Dataset Overview

The Nova Mart dataset follows a **star-schema structure**, where the central fact table
`fact_events` contains the campaign event-level data and is connected to multiple
dimension tables.

The dataset consists of the following tables:

| Table | Type | Purpose |
|---|---|---|
| 📢 `dim_campaigns` | Dimension | Contains campaign-related information |
| 🛍️ `dim_products` | Dimension | Contains product-related information |
| 🏪 `dim_stores` | Dimension | Contains store-related information |
| 📊 `fact_events` | Fact | Contains the actual campaign/event performance data |

---

## ⭐ 2. Data Model

The central table is **`fact_events`**.

It connects the campaign, product, and store information using their respective
key columns.

```text
                    ┌──────────────────────┐
                    │    dim_campaigns     │
                    │──────────────────────│
                    │ campaign_id (PK)     │
                    │ campaign_name        │
                    │ ...                  │
                    └──────────┬───────────┘
                               │
                               │ campaign_id
                               ▼
                    ┌──────────────────────┐
                    │     fact_events      │
                    │──────────────────────│
                    │ campaign_id (FK)     │
                    │ product_code (FK)    │
                    │ store_id (FK)        │
                    │ quantity_sold       │
                    │ base_price          │
                    │ promo_price         │
                    │ ...                  │
                    └───────▲───────▲──────┘
                            │       │
             product_code  │       │  store_id
                            │       │
             ┌──────────────┘       └──────────────┐
             │                                     │
┌────────────┴────────────┐          ┌─────────────┴────────────┐
│     dim_products        │          │       dim_stores         │
│─────────────────────────│          │──────────────────────────│
│ product_code (PK)       │          │ store_id (PK)             │
│ product_category       │          │ city                      │
│ ...                     │          │ ...                       │
└─────────────────────────┘          └──────────────────────────┘

## 📈 3. Important Business Metrics

The analysis focuses on measuring the **impact of promotional campaigns**.

Two important metrics are:

---

### 💰 Incremental Revenue %

**Incremental Revenue % (IR%)** measures the percentage change in revenue after the promotion compared with the revenue before the promotion.

#### 📐 Formula

$$
IR\% =
\frac{Revenue_{after} - Revenue_{before}}
{Revenue_{before}}
\times 100
$$

Where:

$$
Revenue_{before}
=
Base\ Price_{before\ promo}
\times
Quantity\ Sold_{before\ promo}
$$

$$
Revenue_{after}
=
Base\ Price_{after\ promo}
\times
Quantity\ Sold_{after\ promo}
$$

#### 📊 Interpretation

| IR% | Meaning |
|---|---|
| 🟢 **Positive IR%** | Revenue increased after the promotion |
| ⚪ **IR% = 0%** | No change in revenue |
| 🔴 **Negative IR%** | Revenue decreased after the promotion |

---

### 📦 Incremental Sold Units %

**Incremental Sold Units % (ISU%)** measures the percentage change in units sold after the promotion compared with the units sold before the promotion.

#### 📐 Formula

$$
ISU\% =
\frac{Quantity_{sold\ after} - Quantity_{sold\ before}}
{Quantity_{sold\ before}}
\times 100
$$

#### 📊 Interpretation

| ISU% | Meaning |
|---|---|
| 🟢 **Positive ISU%** | More units were sold after the promotion |
| ⚪ **ISU% = 0%** | No change in units sold |
| 🔴 **Negative ISU%** | Fewer units were sold after the promotion |

---

### 💡 Quick Reference

$$
\boxed{
IR\% =
\frac{Revenue_{after} - Revenue_{before}}
{Revenue_{before}}
\times 100
}
$$

$$
\boxed{
ISU\% =
\frac{Quantity_{sold\ after} - Quantity_{sold\ before}}
{Quantity_{sold\ before}}
\times 100
}
$$

###  1. Removing Duplicate Rows

The operations team wants to ensure the **integrity of the events data** by identifying and removing duplicate rows.

Check for and remove duplicate rows in the `events` dataframe based on:

- `store_id`
- `campaign_id`
- `product_code`

### **❓ Question**

**How many duplicate rows were removed?**

In [1]:
# importing necessary libraries

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
df_camp = pd.read_csv(r"C:\Users\Abhinay A\Desktop\resources_1\datasets\dim_campaigns.csv")
df_camp

,campaign_id,campaign_name,start_date,end_date
0,CAMP_DIW_01,Diwali,12-11-2023,18-11-2023
1,CAMP_SAN_01,Sankranti,10-01-2024,16-01-2024


In [4]:
df_pro = pd.read_csv(r"C:\Users\Abhinay A\Desktop\resources_1\datasets\dim_products.csv")
df_pro.head(2)

,product_code,product_name,category
0,P01,Atliq_Masoor_Dal (1KG),Grocery & Staples
1,P02,Atliq_Sonamasuri_Rice (10KG),Grocery & Staples


In [5]:
df_stores = pd.read_csv(r"C:\Users\Abhinay A\Desktop\resources_1\datasets\dim_stores.csv")
df_stores.head(2)

,store_id,city
0,STTRV-0,Trivandrum
1,STMDU-3,Madurai


In [6]:
df_fact_events = pd.read_csv(r"C:\Users\Abhinay A\Desktop\resources_1\datasets\fact_events.csv")
df_fact_events.head(2)

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo)
0,8481be,STCHE-1,CAMP_DIW_01,P04,290,327.0,25% OFF,217,287
1,20618e,STCHE-3,CAMP_SAN_01,P04,370,379.0,BOGOF,185,1622


### Identify Duplicate Events


In [7]:
# Check the number of duplicate rows
duplicate_count = df_fact_events.duplicated(subset=['store_id', 'campaign_id', 'product_code']).sum()

print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 10


### printing duplicate rows

In [8]:
duplicate_count_rows = df_fact_events[df_fact_events.duplicated(keep=False)]
duplicate_count_rows

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo)
40,46d57b,STBLR-9,CAMP_DIW_01,P13,350,94.0,BOGOF,175,329
74,46d57b,STBLR-9,CAMP_DIW_01,P13,350,94.0,BOGOF,175,329
126,4ad12b,STBLR-4,CAMP_DIW_01,P15,3000,407.0,500 Cashback,2500,1245
136,f6aa36,STVSK-3,CAMP_DIW_01,P06,415,63.0,25% OFF,311,54
333,0491f4,STVSK-2,CAMP_DIW_01,P11,190,38.0,50% OFF,95,58
395,6.24E+11,STMLR-0,CAMP_SAN_01,P03,200,206.0,BOGOF,100,541
410,f6aa36,STVSK-3,CAMP_DIW_01,P06,415,63.0,25% OFF,311,54
423,491ff2,STVJD-1,CAMP_SAN_01,P12,62,30.0,50% OFF,31,42
541,6.24E+11,STMLR-0,CAMP_SAN_01,P03,200,206.0,BOGOF,100,541
703,0f8686,STVSK-0,CAMP_SAN_01,P07,300,24.0,BOGOF,150,92


In [9]:
print(len(duplicate_count_rows))

20


### Before duplicates

In [10]:
len(df_fact_events)

1510

### Remove Duplicate Events


In [11]:
df_fact_events = df_fact_events.drop_duplicates(["store_id", "campaign_id", "product_code"])

### After removal of duplicated

In [12]:
len(df_fact_events)

1500

##  2. How many cities have more than 5 stores?

In [13]:
df_stores.columns

Index(['store_id', 'city'], dtype='object')

In [14]:
cities = df_stores['city'].unique()
cities

array(['Trivandrum', 'Madurai', 'Hyderabad', 'Visakhapatnam',
       'Coimbatore', 'Bengaluru', 'Chennai', 'Vijayawada', 'Mysuru',
       'Mangalore'], dtype=object)

In [15]:
## no of stores in each city

city_store_count = df_stores.groupby('city')['store_id'].count()
city_store_count

city
Bengaluru        10
Chennai           8
Coimbatore        5
Hyderabad         7
Madurai           4
Mangalore         3
Mysuru            4
Trivandrum        2
Vijayawada        2
Visakhapatnam     5
Name: store_id, dtype: int64

In [16]:
## cities having more than 5 stores

city_store_count_5 = city_store_count[city_store_count>5]
city_store_count_5

city
Bengaluru    10
Chennai       8
Hyderabad     7
Name: store_id, dtype: int64

##  3. Imputing Missing Values in Quantity Sold

The sales team has identified **missing values** in the `quantity_sold(before_promo)` column.

To handle these missing values, we will:

- 🔍 Identify the missing values.
- 📊 Calculate the **median quantity sold before the promotion**.
- 🧮 Use the median to estimate and fill the missing values.
- 🔢 Count how many missing values were filled.

### Question

**How many missing values were filled, and what is the median used for imputation?**

In [17]:
df_fact_events.columns

Index(['event_id', 'store_id', 'campaign_id', 'product_code',
       'base_price(before_promo)', 'quantity_sold(before_promo)', 'promo_type',
       'base_price(after_promo)', 'quantity_sold(after_promo)'],
      dtype='object')

In [18]:
###Find the number of missing values

missing_val_count = df_fact_events['quantity_sold(before_promo)'].isna().sum()
missing_val_count

20

In [19]:
## printing missing value rows

missing_val_rows = df_fact_events[df_fact_events['quantity_sold(before_promo)'].isna()]
missing_val_rows

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo)
63,70c312,STHYD-4,CAMP_SAN_01,P13,350,NaN,BOGOF,175,534
119,d31787,STMYS-2,CAMP_SAN_01,P10,50,NaN,25% OFF,37,20
141,141d98,STCHE-4,CAMP_SAN_01,P03,200,NaN,BOGOF,100,1695
163,873333,STMLR-0,CAMP_DIW_01,P15,3000,NaN,500 Cashback,2500,509
341,2ef46d,STMDU-0,CAMP_DIW_01,P02,860,NaN,33% OFF,576,430
391,5372de,STCBE-3,CAMP_SAN_01,P10,50,NaN,25% OFF,37,22
558,77435f,STBLR-3,CAMP_SAN_01,P01,172,NaN,33% OFF,115,387
714,a1ef43,STTRV-1,CAMP_SAN_01,P12,62,NaN,50% OFF,31,38
745,95f061,STMYS-3,CAMP_SAN_01,P15,3000,NaN,500 Cashback,2500,443
758,5f313a,STCHE-2,CAMP_DIW_01,P10,65,NaN,50% OFF,32,135


 #### Calculate the median quantity sold before the promotion.

In [20]:
medina_val_count = df_fact_events['quantity_sold(before_promo)'].median()
medina_val_count

78.0

In [21]:
## replacing null values with median value

df_fact_events['quantity_sold(before_promo)'] = df_fact_events['quantity_sold(before_promo)'].fillna(medina_val_count)

In [22]:
## verifying missing values filled or not

df_fact_events['quantity_sold(before_promo)'].isna().sum()

0

### 4.Identify the Product Category with the Lowest Base Price Before the Promotion

In [23]:
### Merge fact_events with dim_products to get the product category

events_products = pd.merge(df_pro, df_fact_events, on='product_code')
events_products.head()

,product_code,product_name,category,event_id,store_id,campaign_id,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo)
0,P01,Atliq_Masoor_Dal (1KG),Grocery & Staples,31c430,STHYD-1,CAMP_SAN_01,172,312.0,33% OFF,115,393
1,P01,Atliq_Masoor_Dal (1KG),Grocery & Staples,029ff8,STMLR-0,CAMP_SAN_01,172,169.0,33% OFF,115,236
2,P01,Atliq_Masoor_Dal (1KG),Grocery & Staples,fad26a,STCHE-3,CAMP_DIW_01,172,369.0,33% OFF,115,546
3,P01,Atliq_Masoor_Dal (1KG),Grocery & Staples,166989,STCBE-1,CAMP_SAN_01,172,169.0,33% OFF,115,206
4,P01,Atliq_Masoor_Dal (1KG),Grocery & Staples,3cb389,STCHE-6,CAMP_DIW_01,172,309.0,33% OFF,115,460


In [24]:
product_categories = events_products.groupby('category')['product_code'].count()
product_categories

category
Combo1               100
Grocery & Staples    400
Home Appliances      200
Home Care            400
Personal Care        400
Name: product_code, dtype: int64

In [25]:
lowest_price_val = events_products['base_price(before_promo)'].min()
lowest_price_val

50

In [26]:
prod_with_lowest_price_val = events_products[events_products['base_price(before_promo)']==lowest_price_val]
prod_with_lowest_price_val.head(3)

,product_code,product_name,category,event_id,store_id,campaign_id,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo)
700,P10,Atliq_Cream_Beauty_Bathing_Soap (125GM),Personal Care,ed7b12,STVSK-0,CAMP_SAN_01,50,16.0,25% OFF,37,13
702,P10,Atliq_Cream_Beauty_Bathing_Soap (125GM),Personal Care,92151c,STMDU-3,CAMP_SAN_01,50,25.0,25% OFF,37,20
704,P10,Atliq_Cream_Beauty_Bathing_Soap (125GM),Personal Care,decd5e,STMDU-2,CAMP_SAN_01,50,21.0,25% OFF,37,17


In [27]:
print(len(prod_with_lowest_price_val))

50


### 5. What is the total quantity sold after the promotion for the BOGOF promo type during the Diwali campaign?

In [28]:
##  Merging df_camp information with df_fact_events

event_campaigns = pd.merge(df_fact_events, df_camp, on='campaign_id')
event_campaigns.head(3)

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo),campaign_name,start_date,end_date
0,8481be,STCHE-1,CAMP_DIW_01,P04,290,327.0,25% OFF,217,287,Diwali,12-11-2023,18-11-2023
1,20618e,STCHE-3,CAMP_SAN_01,P04,370,379.0,BOGOF,185,1622,Sankranti,10-01-2024,16-01-2024
2,f30579,STBLR-9,CAMP_DIW_01,P02,860,337.0,33% OFF,576,488,Diwali,12-11-2023,18-11-2023


In [29]:
##  Filter Diwali + BOGOF and calculate total quantity sold after promotion

total_quantity_sold_after_pro = event_campaigns[(event_campaigns['campaign_name']=='Diwali') & (event_campaigns['promo_type']=='BOGOF')]
total_quantity_sold_after_pro.head(3)

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo),campaign_name,start_date,end_date
6,6bbadf,STHYD-1,CAMP_DIW_01,P14,1020,43.0,BOGOF,510,127,Diwali,12-11-2023,18-11-2023
8,66c422,STBLR-5,CAMP_DIW_01,P07,300,66.0,BOGOF,150,227,Diwali,12-11-2023,18-11-2023
9,108d5a,STCBE-0,CAMP_DIW_01,P14,1020,29.0,BOGOF,510,101,Diwali,12-11-2023,18-11-2023


In [30]:
total_quantity_sold_after_pro['quantity_sold(after_promo)'].sum()

34461

## 6. Which Store Recorded the Highest Quantity Sold After the Promotion During the Diwali Campaign?

In [31]:
stores = event_campaigns['store_id'].unique()
stores

array(['STCHE-1', 'STCHE-3', 'STBLR-9', 'STBLR-7', 'STHYD-5', 'STCHE-6',
       'STHYD-1', 'STVJD-0', 'STBLR-5', 'STCBE-0', 'STBLR-1', 'STVSK-0',
       'STCHE-5', 'STBLR-8', 'STTRV-1', 'STHYD-0', 'STVSK-3', 'STCHE-4',
       'STMDU-3', 'STCBE-1', 'STVSK-1', 'STMDU-2', 'STMLR-2', 'STBLR-0',
       'STMLR-1', 'STMLR-0', 'STMYS-0', 'STCBE-2', 'STCHE-2', 'STMYS-2',
       'STHYD-6', 'STMDU-0', 'STCBE-3', 'STMYS-3', 'STHYD-4', 'STVSK-2',
       'STBLR-6', 'STBLR-3', 'STBLR-2', 'STCHE-7', 'STBLR-4', 'STTRV-0',
       'STHYD-2', 'STVSK-4', 'STMYS-1', 'STCBE-4', 'STVJD-1', 'STCHE-0',
       'STMDU-1', 'STHYD-3'], dtype=object)

In [32]:
df_camp.head(2)

,campaign_id,campaign_name,start_date,end_date
0,CAMP_DIW_01,Diwali,12-11-2023,18-11-2023
1,CAMP_SAN_01,Sankranti,10-01-2024,16-01-2024


In [33]:
df_stores.head(2)

,store_id,city
0,STTRV-0,Trivandrum
1,STMDU-3,Madurai


In [34]:
df_fact_events.head(2)

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo)
0,8481be,STCHE-1,CAMP_DIW_01,P04,290,327.0,25% OFF,217,287
1,20618e,STCHE-3,CAMP_SAN_01,P04,370,379.0,BOGOF,185,1622


In [35]:
## merging df_camp and df_fact_events

events_camp_stores = pd.merge(df_fact_events, df_camp, on='campaign_id')
events_camp_stores.head(3)

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo),campaign_name,start_date,end_date
0,8481be,STCHE-1,CAMP_DIW_01,P04,290,327.0,25% OFF,217,287,Diwali,12-11-2023,18-11-2023
1,20618e,STCHE-3,CAMP_SAN_01,P04,370,379.0,BOGOF,185,1622,Sankranti,10-01-2024,16-01-2024
2,f30579,STBLR-9,CAMP_DIW_01,P02,860,337.0,33% OFF,576,488,Diwali,12-11-2023,18-11-2023


In [36]:
## filtering diwali campaign

diwali_camp = events_camp_stores[events_camp_stores['campaign_name']=='Diwali']
diwali_camp.head(2)

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo),campaign_name,start_date,end_date
0,8481be,STCHE-1,CAMP_DIW_01,P04,290,327.0,25% OFF,217,287,Diwali,12-11-2023,18-11-2023
2,f30579,STBLR-9,CAMP_DIW_01,P02,860,337.0,33% OFF,576,488,Diwali,12-11-2023,18-11-2023


In [37]:
total_qua_sold_each_store = diwali_camp.groupby('store_id')['quantity_sold(after_promo)'].sum()
total_qua_sold_each_store

store_id
STBLR-0    4759
STBLR-1    4146
STBLR-2    4169
STBLR-3    4373
STBLR-4    4408
STBLR-5    4193
STBLR-6    4857
STBLR-7    4893
STBLR-8    4395
STBLR-9    4186
STCBE-0    3381
STCBE-1    2942
STCBE-2    3093
STCBE-3    3077
STCBE-4    2907
STCHE-0    4100
STCHE-1    3595
STCHE-2    4093
STCHE-3    4605
STCHE-4    5013
STCHE-5    4052
STCHE-6    4445
STCHE-7    4779
STHYD-0    4460
STHYD-1    3778
STHYD-2    4266
STHYD-3    4272
STHYD-4    4227
STHYD-5    4346
STHYD-6    4153
STMDU-0    3545
STMDU-1    3266
STMDU-2    2981
STMDU-3    3071
STMLR-0    2027
STMLR-1    2151
STMLR-2    2138
STMYS-0    3543
STMYS-1    4779
STMYS-2    4130
STMYS-3    4347
STTRV-0    2180
STTRV-1    2072
STVJD-0    2392
STVJD-1    2312
STVSK-0    3005
STVSK-1    3078
STVSK-2    2860
STVSK-3    2656
STVSK-4    2908
Name: quantity_sold(after_promo), dtype: int64

In [38]:
highest_qua_sold = total_qua_sold_each_store.sort_values(ascending=False)
highest_qua_sold

store_id
STCHE-4    5013
STBLR-7    4893
STBLR-6    4857
STMYS-1    4779
STCHE-7    4779
STBLR-0    4759
STCHE-3    4605
STHYD-0    4460
STCHE-6    4445
STBLR-4    4408
STBLR-8    4395
STBLR-3    4373
STMYS-3    4347
STHYD-5    4346
STHYD-3    4272
STHYD-2    4266
STHYD-4    4227
STBLR-5    4193
STBLR-9    4186
STBLR-2    4169
STHYD-6    4153
STBLR-1    4146
STMYS-2    4130
STCHE-0    4100
STCHE-2    4093
STCHE-5    4052
STHYD-1    3778
STCHE-1    3595
STMDU-0    3545
STMYS-0    3543
STCBE-0    3381
STMDU-1    3266
STCBE-2    3093
STVSK-1    3078
STCBE-3    3077
STMDU-3    3071
STVSK-0    3005
STMDU-2    2981
STCBE-1    2942
STVSK-4    2908
STCBE-4    2907
STVSK-2    2860
STVSK-3    2656
STVJD-0    2392
STVJD-1    2312
STTRV-0    2180
STMLR-1    2151
STMLR-2    2138
STTRV-1    2072
STMLR-0    2027
Name: quantity_sold(after_promo), dtype: int64

In [39]:
highest_qua_sold.head(1)

store_id
STCHE-4    5013
Name: quantity_sold(after_promo), dtype: int64

##  7. Compare Sales Performance: Sankranti vs Diwali

Understand which campaign had the **most successful outcome** by comparing the total quantity sold **before and after** the promotions.

###  Question

**Which campaign saw a greater increase in sales — Sankranti or Diwali?**

The key calculation is:

Increase=Total After Promo−Total Before Promo

In [40]:
df_camp.head(2)

,campaign_id,campaign_name,start_date,end_date
0,CAMP_DIW_01,Diwali,12-11-2023,18-11-2023
1,CAMP_SAN_01,Sankranti,10-01-2024,16-01-2024


In [41]:
df_fact_events.head(2)

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo)
0,8481be,STCHE-1,CAMP_DIW_01,P04,290,327.0,25% OFF,217,287
1,20618e,STCHE-3,CAMP_SAN_01,P04,370,379.0,BOGOF,185,1622


In [42]:
events_campaigns = pd.merge(df_fact_events,df_camp, on='campaign_id')
events_campaigns.head(2)

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo),campaign_name,start_date,end_date
0,8481be,STCHE-1,CAMP_DIW_01,P04,290,327.0,25% OFF,217,287,Diwali,12-11-2023,18-11-2023
1,20618e,STCHE-3,CAMP_SAN_01,P04,370,379.0,BOGOF,185,1622,Sankranti,10-01-2024,16-01-2024


In [43]:
## Grouping by campaigns and calculating quantities

campaign_sales = events_campaigns.groupby('campaign_name').agg(total_before=('quantity_sold(before_promo)', 'sum'),total_after=('quantity_sold(after_promo)', 'sum'))
campaign_sales.head(2)

,total_before,total_after
campaign_name,,
Diwali,109756.0,183404
Sankranti,97894.0,252069


In [44]:
## calculating sales increase

campaign_sales['increase'] = (campaign_sales['total_after'] - campaign_sales['total_before'])
campaign_sales

,total_before,total_after,increase
campaign_name,,,
Diwali,109756.0,183404,73648.0
Sankranti,97894.0,252069,154175.0


## 8. Identify the Product with the Highest Incremental Revenue % (IR%) During the Sankranti Campaign

Determine which product recorded the **highest Incremental Revenue Percentage (IR%)** during the Sankranti campaign.

### Question

**Which product recorded the highest IR% during the Sankranti campaign, and what was its IR%?**

### Incremental Revenue Percentage (IR%)

For each product, we need to calculate the **Incremental Revenue Percentage (IR%)** to measure the change in revenue after the promotion.

#### Formula

$$
IR\% =
\frac{Revenue_{after} - Revenue_{before}}
{Revenue_{before}}
\times 100
$$

#### Revenue Before Promotion

$$
Revenue_{before}
=
Base\ Price_{before\ promo}
\times
Quantity\ Sold_{before\ promo}
$$

#### Revenue After Promotion

$$
Revenue_{after}
=
Base\ Price_{after\ promo}
\times
Quantity\ Sold_{after\ promo}
$$

> ** Interpretation:** A higher positive IR% indicates a greater increase in revenue after the promotion compared with before the promotion.

In [45]:
df_camp.head(1)

,campaign_id,campaign_name,start_date,end_date
0,CAMP_DIW_01,Diwali,12-11-2023,18-11-2023


In [46]:
df_pro.head(1)

,product_code,product_name,category
0,P01,Atliq_Masoor_Dal (1KG),Grocery & Staples


In [47]:
df_fact_events.head(1)

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo)
0,8481be,STCHE-1,CAMP_DIW_01,P04,290,327.0,25% OFF,217,287


In [48]:
## merging df_pro and df_fact_events

events_data = pd.merge(df_fact_events, df_pro, on = 'product_code').merge(df_camp,on='campaign_id')
events_data.head(3)

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo),product_name,category,campaign_name,start_date,end_date
0,8481be,STCHE-1,CAMP_DIW_01,P04,290,327.0,25% OFF,217,287,Atliq_Farm_Chakki_Atta (1KG),Grocery & Staples,Diwali,12-11-2023,18-11-2023
1,20618e,STCHE-3,CAMP_SAN_01,P04,370,379.0,BOGOF,185,1622,Atliq_Farm_Chakki_Atta (1KG),Grocery & Staples,Sankranti,10-01-2024,16-01-2024
2,f30579,STBLR-9,CAMP_DIW_01,P02,860,337.0,33% OFF,576,488,Atliq_Sonamasuri_Rice (10KG),Grocery & Staples,Diwali,12-11-2023,18-11-2023


In [49]:
## filtering sankranti data

sankranti_camp = events_data[events_data['campaign_name'] == 'Sankranti'].copy()

In [50]:
##  Calculate revenue before promotion

sankranti_camp['revenue_before'] = sankranti_camp['base_price(before_promo)'] * sankranti_camp['quantity_sold(before_promo)']

In [51]:
##  Calculate revenue after promotion

sankranti_camp['revenue_after'] = sankranti_camp['base_price(after_promo)'] * sankranti_camp['quantity_sold(after_promo)']

In [52]:
# Calculate IR%

sankranti_camp['IR%'] = ((sankranti_camp['revenue_after'] -sankranti_camp['revenue_before'])/ sankranti_camp['revenue_before']) * 100

In [53]:
sankranti_camp.head(3)

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo),product_name,category,campaign_name,start_date,end_date,revenue_before,revenue_after,IR%
1,20618e,STCHE-3,CAMP_SAN_01,P04,370,379.0,BOGOF,185,1622,Atliq_Farm_Chakki_Atta (1KG),Grocery & Staples,Sankranti,10-01-2024,16-01-2024,140230.0,300070,113.984169
4,6d153f,STHYD-5,CAMP_SAN_01,P15,3000,122.0,500 Cashback,2500,272,Atliq_Home_Essential_8_Product_Combo,Combo1,Sankranti,10-01-2024,16-01-2024,366000.0,680000,85.792350
7,6.88E+10,STVJD-0,CAMP_SAN_01,P08,1190,22.0,BOGOF,595,88,Atliq_Double_Bedsheet_set,Home Care,Sankranti,10-01-2024,16-01-2024,26180.0,52360,100.000000


In [54]:
highest_IR = sankranti_camp.sort_values('IR%',ascending=False).head(1)
highest_IR

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo),product_name,category,campaign_name,start_date,end_date,revenue_before,revenue_after,IR%
140,141d98,STCHE-4,CAMP_SAN_01,P03,200,78.0,BOGOF,100,1695,Atliq_Suflower_Oil (1L),Grocery & Staples,Sankranti,10-01-2024,16-01-2024,15600.0,169500,986.538462


In [55]:
print(highest_IR[['product_name', 'IR%']])

                product_name         IR%
140  Atliq_Suflower_Oil (1L)  986.538462


In [76]:
# Merge campaign and product information
events_data = df_fact_events.merge(
    df_camp[['campaign_id', 'campaign_name']],
    on='campaign_id',
    how='left'
).merge(
    df_pro[['product_code', 'product_name']],
    on='product_code',
    how='left'
)

# Filter Sankranti campaign
sankranti_data = events_data[
    events_data['campaign_name'] == 'Sankranti'
].copy()

# Calculate revenue before promotion
sankranti_data['revenue_before'] = (
    sankranti_data['base_price(before_promo)'] *
    sankranti_data['quantity_sold(before_promo)']
)

# Calculate revenue after promotion
sankranti_data['revenue_after'] = (
    sankranti_data['base_price(after_promo)'] *
    sankranti_data['quantity_sold(after_promo)']
)

# Aggregate revenue for each product
product_revenue = sankranti_data.groupby(
    ['product_code', 'product_name']
).agg(
    revenue_before=('revenue_before', 'sum'),
    revenue_after=('revenue_after', 'sum')
).reset_index()

# Calculate Incremental Revenue Percentage
product_revenue['IR%'] = (
    (product_revenue['revenue_after'] -
     product_revenue['revenue_before'])
    / product_revenue['revenue_before']
) * 100

# Find the product with the highest IR%
result = product_revenue.sort_values(
    'IR%',
    ascending=False
).head(1)

# Display result
print(result[['product_name', 'IR%']])

              product_name        IR%
2  Atliq_Suflower_Oil (1L)  91.826561


In [77]:
print(
    result['product_name'].iloc[0],
    round(result['IR%'].iloc[0], 2),
    sep=','
)

Atliq_Suflower_Oil (1L),91.83


## 9. Identify the Store with the Lowest Incremental Sold Units % (ISU%)

During the **Diwali campaign**, identify the store in **Visakhapatnam** that recorded the lowest **Incremental Sold Units Percentage (ISU%)**.

### Question

**Which store in Visakhapatnam recorded the lowest ISU% during the Diwali campaign, and what was its ISU%?**

### Incremental Sold Units Percentage (ISU%)

The **Incremental Sold Units Percentage (ISU%)** measures the percentage change in the quantity of units sold after the promotion compared with before the promotion.

####  Formula

$$
ISU\% =
\frac{Quantity_{after} - Quantity_{before}}
{Quantity_{before}}
\times 100
$$

Where:

$$
Quantity_{before}
=
Quantity\ Sold_{before\ promo}
$$

$$
Quantity_{after}
=
Quantity\ Sold_{after\ promo}
$$

#### Interpretation

- 🟢 **Positive ISU%** → Quantity sold increased after the promotion.
- ⚪ **ISU% = 0%** → No change in quantity sold.
- 🔴 **Negative ISU%** → Quantity sold decreased after the promotion.

In [56]:
df_stores.head(1)

,store_id,city
0,STTRV-0,Trivandrum


In [57]:
df_stores.city.unique()

array(['Trivandrum', 'Madurai', 'Hyderabad', 'Visakhapatnam',
       'Coimbatore', 'Bengaluru', 'Chennai', 'Vijayawada', 'Mysuru',
       'Mangalore'], dtype=object)

In [58]:
df_camp.head(1)

,campaign_id,campaign_name,start_date,end_date
0,CAMP_DIW_01,Diwali,12-11-2023,18-11-2023


In [59]:
df_fact_events.head(1)

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo)
0,8481be,STCHE-1,CAMP_DIW_01,P04,290,327.0,25% OFF,217,287


In [60]:
## merging the tables

In [61]:
events_data = pd.merge(df_fact_events, df_stores,on='store_id').merge(df_camp,on='campaign_id')
events_data.head(2)

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo),city,campaign_name,start_date,end_date
0,8481be,STCHE-1,CAMP_DIW_01,P04,290,327.0,25% OFF,217,287,Chennai,Diwali,12-11-2023,18-11-2023
1,20618e,STCHE-3,CAMP_SAN_01,P04,370,379.0,BOGOF,185,1622,Chennai,Sankranti,10-01-2024,16-01-2024


In [62]:
## filtering Visakhapatnam data during diwali

visakhapatnam_diwali = events_data[(events_data['city'] == 'Visakhapatnam') & (events_data['campaign_name'] == 'Diwali')]
visakhapatnam_diwali.head(2)

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo),city,campaign_name,start_date,end_date
27,ba86f4,STVSK-1,CAMP_DIW_01,P13,350,61.0,BOGOF,175,204,Visakhapatnam,Diwali,12-11-2023,18-11-2023
64,9779b0,STVSK-2,CAMP_DIW_01,P10,65,85.0,50% OFF,32,128,Visakhapatnam,Diwali,12-11-2023,18-11-2023


In [63]:
# Calculate total units before and after for each store

store_sales = visakhapatnam_diwali.groupby('store_id').agg(quantity_before=('quantity_sold(before_promo)', 'sum'),quantity_after=('quantity_sold(after_promo)', 'sum')).reset_index()

In [64]:
# Calculate ISU%

store_sales['ISU%'] = ((store_sales['quantity_after'] - store_sales['quantity_before'])/ store_sales['quantity_before']) * 100
store_sales['ISU%']

0    69.966063
1    61.744614
2    68.136390
3    49.213483
4    50.986501
Name: ISU%, dtype: float64

In [65]:
# Find the store with the lowest ISU%

lowest_ISU = store_sales.sort_values('ISU%').head(1)

print(lowest_ISU)

  store_id  quantity_before  quantity_after       ISU%
3  STVSK-3           1780.0            2656  49.213483


## 10. Identify the Promo Type with Negative IR% and ISU% During the Sankranti Campaign

Determine which **promo type** recorded both a **negative Incremental Revenue Percentage (IR%)** and a **negative Incremental Sold Units Percentage (ISU%)** during the **Sankranti campaign**.

### Question

**Which promo type had both a negative IR% and ISU% during the Sankranti campaign?**

We need to identify the promo type during the Sankranti campaign for which both:

IR% < 0
ISU% < 0

The safest approach is to first aggregate revenue and quantity at the promo-type level, then calculate the two percentages.

In [66]:
## Merge campaign information

events_campaign = pd.merge(df_fact_events,df_camp,on='campaign_id')

In [67]:
## Filter Sankranti

sankranti_data = events_campaigns[events_campaigns['campaign_name'] == 'Sankranti'].copy()

In [68]:
## Calculate revenue before and after

sankranti_data['revenue_before'] = (sankranti_data['base_price(before_promo)'] *sankranti_data['quantity_sold(before_promo)'])

sankranti_data['revenue_after'] = (sankranti_data['base_price(after_promo)'] *sankranti_data['quantity_sold(after_promo)'])

In [69]:
sankranti_data.head(2)

,event_id,store_id,campaign_id,product_code,base_price(before_promo),quantity_sold(before_promo),promo_type,base_price(after_promo),quantity_sold(after_promo),campaign_name,start_date,end_date,revenue_before,revenue_after
1,20618e,STCHE-3,CAMP_SAN_01,P04,370,379.0,BOGOF,185,1622,Sankranti,10-01-2024,16-01-2024,140230.0,300070
4,6d153f,STHYD-5,CAMP_SAN_01,P15,3000,122.0,500 Cashback,2500,272,Sankranti,10-01-2024,16-01-2024,366000.0,680000


In [70]:
## Aggregate by promo type

promo_performance = sankranti_data.groupby('promo_type').agg(
    revenue_before=('revenue_before', 'sum'),
    revenue_after=('revenue_after', 'sum'),
    quantity_before=('quantity_sold(before_promo)', 'sum'),
    quantity_after=('quantity_sold(after_promo)', 'sum')
).reset_index()

In [71]:
## Calculate IR% and ISU%

promo_performance['IR%'] = (
    (promo_performance['revenue_after'] -
     promo_performance['revenue_before'])
    / promo_performance['revenue_before']
) * 100

promo_performance['ISU%'] = (
    (promo_performance['quantity_after'] -
     promo_performance['quantity_before'])
    / promo_performance['quantity_before']
) * 100

In [72]:
promo_performance.head(2)

,promo_type,revenue_before,revenue_after,quantity_before,quantity_after,IR%,ISU%
0,25% OFF,935195.0,567387,6601.0,5307,-39.329552,-19.603090
1,33% OFF,20483136.0,19273955,33624.0,47459,-5.903300,41.146205


In [73]:
## Find the promo type satisfying both conditions

result = promo_performance[
    (promo_performance['IR%'] < 0) &
    (promo_performance['ISU%'] < 0)]

In [74]:
result[['promo_type', 'IR%', 'ISU%']]

,promo_type,IR%,ISU%
0,25% OFF,-39.329552,-19.60309


In [78]:
# Merge campaign information with fact events
events_data = df_fact_events.merge(
    df_camp[['campaign_id', 'campaign_name']],
    on='campaign_id',
    how='left'
)

# Filter only Sankranti campaign
sankranti_data = events_data[
    events_data['campaign_name'] == 'Sankranti'
].copy()

# Calculate revenue before promotion
sankranti_data['revenue_before'] = (
    sankranti_data['base_price(before_promo)'] *
    sankranti_data['quantity_sold(before_promo)']
)

# Calculate revenue after promotion
sankranti_data['revenue_after'] = (
    sankranti_data['base_price(after_promo)'] *
    sankranti_data['quantity_sold(after_promo)']
)

# Aggregate revenue and quantity by promo type
promo_performance = sankranti_data.groupby('promo_type').agg(
    revenue_before=('revenue_before', 'sum'),
    revenue_after=('revenue_after', 'sum'),
    quantity_before=('quantity_sold(before_promo)', 'sum'),
    quantity_after=('quantity_sold(after_promo)', 'sum')
).reset_index()

# Calculate IR%
promo_performance['IR%'] = (
    (promo_performance['revenue_after'] -
     promo_performance['revenue_before'])
    / promo_performance['revenue_before']
) * 100

# Calculate ISU%
promo_performance['ISU%'] = (
    (promo_performance['quantity_after'] -
     promo_performance['quantity_before'])
    / promo_performance['quantity_before']
) * 100

# Find the promo type where both IR% and ISU% are negative
result = promo_performance[
    (promo_performance['IR%'] < 0) &
    (promo_performance['ISU%'] < 0)
]

# Print in the exact format required
for _, row in result.iterrows():
    print(
        f"{row['promo_type']},{row['IR%']:.2f},{row['ISU%']:.2f}"
    )

25% OFF,-39.33,-19.60
